# 04 — Análise Estratégica
**Tech Challenge Fase 3 — FIAP IA Scientist**

Este notebook responde as perguntas de negócio do enunciado usando
o modelo treinado para gerar inteligência aplicável a políticas públicas.

## Perguntas respondidas

1. Quais municípios apresentam maior risco educacional?
2. Quais regiões possuem padrões semelhantes?
3. Como prever municípios que podem não atingir metas futuras?
4. Quais variáveis têm maior influência no modelo?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import silhouette_score

import warnings
warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8")

## 1. Carregamento do modelo e dados

In [ ]:
pipeline = joblib.load("models/modelo_final.joblib")
with open("models/metadata.json", encoding="utf-8") as f:
    metadata = json.load(f)

threshold = metadata["threshold"]
features_num = metadata["features_numericas"]
features_cat = metadata["features_categoricas"]

caminho = Path("data/processed/dataset_enriquecido_v2.parquet")
if not caminho.exists():
    caminho = Path("data/processed/dataset_modelagem_v2.parquet")
df = pd.read_parquet(caminho)

features_all = [f for f in features_num + features_cat if f in df.columns]
X = df[features_all]
y = df["em_risco_2024"]

print(f"Dataset: {df.shape}")
print(f"Modelo: {metadata['modelo']}")
print(f"Threshold: {threshold:.3f}")

## Q1 — Municípios com maior risco educacional

In [ ]:
prob_risco = pipeline.predict_proba(X)[:, 1]

REGIOES = {
    11: "Norte", 12: "Norte", 13: "Norte", 14: "Norte", 15: "Norte",
    16: "Norte", 17: "Norte", 21: "Nordeste", 22: "Nordeste",
    23: "Nordeste", 24: "Nordeste", 25: "Nordeste", 26: "Nordeste",
    27: "Nordeste", 28: "Nordeste", 29: "Nordeste", 31: "Sudeste",
    32: "Sudeste", 33: "Sudeste", 35: "Sudeste", 41: "Sul",
    42: "Sul", 43: "Sul", 50: "Centro-Oeste", 51: "Centro-Oeste",
    52: "Centro-Oeste", 53: "Centro-Oeste"
}

df_rank = df[["id_municipio", "taxa_alf_2023", "em_risco_2024"]].copy()
df_rank["prob_risco"] = prob_risco
df_rank["cod_uf"] = df_rank["id_municipio"].astype(str).str[:2].astype(int)
df_rank["regiao"] = df_rank["cod_uf"].map(REGIOES)
df_rank["nivel_risco"] = pd.cut(
    prob_risco,
    bins=[0, 0.3, 0.5, 0.7, 1.0],
    labels=["Baixo", "Moderado", "Alto", "Critico"]
)

print("Distribuicao por nivel de risco:")
print(df_rank["nivel_risco"].value_counts())

print("
Top 10 municipios de maior risco:")
print(df_rank.nlargest(10, "prob_risco")[
    ["id_municipio", "regiao", "taxa_alf_2023", "prob_risco"]
].round(3).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(prob_risco, bins=30, color="steelblue",
             edgecolor="white", alpha=0.85)
axes[0].axvline(x=threshold, color="red", linestyle="--",
                linewidth=2, label=f"Threshold ({threshold:.2f})")
axes[0].set_xlabel("Probabilidade de Risco")
axes[0].set_ylabel("Municipios")
axes[0].set_title("Distribuicao de Probabilidades")
axes[0].legend()

risco_regiao = df_rank.groupby("regiao")["prob_risco"].mean().sort_values()
cores = ["#d32f2f" if v > 0.5 else "#ff9800" if v > 0.4
         else "#4caf50" for v in risco_regiao.values]
axes[1].barh(risco_regiao.index, risco_regiao.values, color=cores, alpha=0.85)
axes[1].axvline(x=0.5, color="red", linestyle="--", linewidth=1)
axes[1].set_xlabel("Probabilidade Media de Risco")
axes[1].set_title("Risco Medio por Regiao")

plt.tight_layout()
plt.show()

## Q2 — Clustering por perfil educacional

In [ ]:
features_cluster = [f for f in ["taxa_alf_2023", "media_pt_2023", "particip_2023"]
                    if f in df.columns]

df_cluster = df[features_cluster].dropna()
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(df_cluster)

# Silhueta para k otimo
scores = {k: silhouette_score(X_scaled,
           KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_scaled))
          for k in range(2, 6)}
melhor_k = max(scores, key=scores.get)
print(f"Melhor k: {melhor_k} (silhueta: {scores[melhor_k]:.3f})")

kmeans = KMeans(n_clusters=melhor_k, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

df_result = df.loc[df_cluster.index].copy()
df_result["cluster"] = clusters

perfil = df_result.groupby("cluster")[features_cluster].mean().round(2)
print("
Perfil medio por cluster:")
print(perfil.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(
    df_result["taxa_alf_2023"],
    df_result["media_pt_2023"],
    c=df_result["cluster"], cmap="RdYlGn", alpha=0.4, s=15
)
ax.set_xlabel("Taxa de Alfabetizacao 2023 (%)")
ax.set_ylabel("Media Portugues 2023")
ax.set_title(f"Clustering de Municipios (K-Means, k={melhor_k})")
plt.colorbar(scatter, label="Cluster")
plt.tight_layout()
plt.show()

## Q3 — Projecao ate 2030

In [ ]:
if "meta_2030" in df.columns:
    df_proj = df[["id_municipio", "taxa_alf_2023", "meta_2030"]].dropna().copy()
    variacao_anual = 2.36
    anos = 2030 - 2024
    df_proj["taxa_projetada_2030"] = df_proj["taxa_alf_2023"] + variacao_anual * (anos + 1)
    df_proj["atingira_meta"] = df_proj["taxa_projetada_2030"] >= df_proj["meta_2030"]

    total = len(df_proj)
    atingira = df_proj["atingira_meta"].sum()
    print(f"Municipios que atingirao a meta: {atingira} ({atingira/total*100:.1f}%)")
    print(f"Municipios que NAO atingirao:    {total-atingira} ({(total-atingira)/total*100:.1f}%)")

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.scatter(df_proj["taxa_alf_2023"], df_proj["taxa_projetada_2030"],
               c=df_proj["atingira_meta"].map({True: "#4caf50", False: "#d32f2f"}),
               alpha=0.4, s=10)
    ax.axhline(y=80, color="red", linestyle="--", linewidth=2, label="Meta 2030 (80%)")
    ax.set_xlabel("Taxa 2023 (%)")
    ax.set_ylabel("Taxa Projetada 2030 (%)")
    ax.set_title("Projecao de Municipios ate 2030")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Coluna meta_2030 nao disponivel no dataset")

## Conclusoes Estrategicas

- Nordeste e Norte concentram os municipios de maior risco
- Clustering identifica 4 perfis: Critico, Vulneravel, Em desenvolvimento, Avancado
- Com a variacao media observada (2.36 pts/ano), parte significativa dos municipios
  nao atingira a meta de 80% ate 2030 sem intervencao
- Lista de priorizacao gerada em reports/priorizacao_municipios_v2.csv

### Recomendacoes
1. Intervencao imediata nos municipios com prob_risco > 0.7
2. Foco em lingua portuguesa — preditor dominante do modelo
3. Atencao especial ao Norte e Nordeste — gap estrutural confirmado
4. Monitorar municipios com queda de participacao — sinal de deterioracao